In [1]:
import json
import requests
from pathlib import Path

import duckdb
import polyline
import pandas as pd

from geopy.distance import geodesic

from to_gpx import to_gpx

In [2]:
DATA_BASE_PATH = Path("./data").resolve().absolute()
GRAPHHOPPER_BASE_URL = "http://localhost:8989"
GPS_ACCURACY = 50

In [ ]:
gps_data_path = DATA_BASE_PATH / "gps_data.parquet"
ground_truth_path = DATA_BASE_PATH / "ground_truth_route.parquet"
newson_krumm_route_network_path = DATA_BASE_PATH / "road_network.parquet"

In [4]:
ground_truth_df = duckdb.query(
    f"""
    WITH gt AS (
        SELECT edge_id, traversed, ROW_NUMBER() OVER () as row_num
        FROM '{ground_truth_path}'
    )
    SELECT gt.edge_id, traversed, linestring
    FROM gt
    LEFT JOIN '{newson_krumm_route_network_path}' nkr 
    ON gt.edge_id = nkr.edge_id
    ORDER BY gt.row_num
    """
).to_df()
ground_truth_df

,edge_id,traversed,linestring
0,884147800801,1,"LINESTRING(-122.109748721123 47.6673012971878,..."
1,884147800802,1,"LINESTRING(-122.105398178101 47.6675292849541,..."
2,884147800421,1,"LINESTRING(-122.102048099041 47.6676607131958,..."
3,884147800422,1,"LINESTRING(-122.1028393507 47.6681300997734, -..."
4,884147800423,1,"LINESTRING(-122.103689610958 47.6685512065887,..."
...,...,...,...
571,884147801154,1,"LINESTRING(-122.14302957058 47.6376816630363, ..."
572,884147800845,1,"LINESTRING(-122.14302957058 47.6378801465034, ..."
573,884147800842,1,"LINESTRING(-122.14291960001 47.6386311650276, ..."
574,884147800843,1,"LINESTRING(-122.142908871174 47.6404094696045,..."


In [5]:
route_network_df = duckdb.query(
    f"""
    SELECT edge_id, linestring 
    FROM '{newson_krumm_route_network_path}'
    """
).to_df()
route_network_df

,edge_id,linestring
0,883991900000,"LINESTRING(-122.732318937778 47.8899192810059,..."
1,883991900001,"LINESTRING(-122.71107852459 47.8776508569717, ..."
2,883991900002,"LINESTRING(-122.707419991493 47.8761515021324,..."
3,883991900003,"LINESTRING(-122.707419991493 47.8761515021324,..."
4,883991900004,"LINESTRING(-122.715329825878 47.8818699717522,..."
...,...,...
158162,884152400184,"LINESTRING(-121.777908504009 47.4525502324104,..."
158163,884152400185,"LINESTRING(-121.781320273876 47.4532207846642,..."
158164,884152400186,"LINESTRING(-121.784308254719 47.4577805399895,..."
158165,884152400187,"LINESTRING(-121.782489717007 47.4503803253174,..."


In [6]:
def parse_linestring(linestring_series: pd.Series) -> pd.Series:
    track_segs = linestring_series.str.replace(r"^LINESTRING\(|\)$", "", regex=True)
    track_segs = track_segs.str.replace(",", ";").replace(r"\s+", " ", regex=True)
    track_segs = track_segs.str.split(";")
    track_segs = track_segs.apply(
        lambda x: [
            tuple((float(lat), float(lon)))
            for (lon, lat) in (point.split() for point in x)
        ]
    )
    return track_segs

In [7]:
ground_truth_df["track_segs"] = parse_linestring(ground_truth_df["linestring"])
route_network_df["track_segs"] = parse_linestring(route_network_df["linestring"])

In [8]:
str(route_network_df[["edge_id", "track_segs"]].head())

'        edge_id                                         track_segs\n0  883991900000  [(47.8899192810059, -122.732318937778), (47.89...\n1  883991900001  [(47.8776508569717, -122.71107852459), (47.878...\n2  883991900002  [(47.8761515021324, -122.707419991493), (47.87...\n3  883991900003  [(47.8761515021324, -122.707419991493), (47.87...\n4  883991900004  [(47.8818699717522, -122.715329825878), (47.88...'

In [9]:
gps_df = duckdb.query(f"SELECT * FROM '{gps_data_path}'").to_df()
gps_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7531 entries, 0 to 7530
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   recorded_timestamp  7531 non-null   datetime64[us]
 1   lon                 7531 non-null   float64       
 2   lat                 7531 non-null   float64       
dtypes: datetime64[us](1), float64(2)
memory usage: 176.6 KB


In [10]:
gps_df.head()

,recorded_timestamp,lon,lat
0,2009-01-17 20:27:37,-122.107083,47.667483
1,2009-01-17 20:27:38,-122.107067,47.667500
2,2009-01-17 20:27:39,-122.107067,47.667500
3,2009-01-17 20:27:40,-122.107033,47.667517
4,2009-01-17 20:27:41,-122.106983,47.667533


In [11]:
points = gps_df[["lon", "lat", "recorded_timestamp"]].to_numpy()

gpx_path = to_gpx(points, DATA_BASE_PATH / "gps.gpx")

In [12]:
def request_map_matching(point_gpx_path: Path, output_path: Path) -> str:
    url = f"{GRAPHHOPPER_BASE_URL}/match?profile=car&gps_accuracy={GPS_ACCURACY}&type=json&locale=pt_BR&details=osm_way_id"
    headers = {
        "Content-Type": "application/gpx+xml",
    }

    with open(point_gpx_path, "rb") as f:
        body = f.read()

    req = requests.post(
        url,
        headers=headers,
        data=body,
    )

    req.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(req.content)

    print(f"Map matching result saved to {output_path}")

    res =  json.loads(req.content)["paths"][0]

    return {
        "points": res["points"],
        "edge_ids": res["details"]["osm_way_id"],
    }

In [13]:
graphhopper_result = request_map_matching(
    point_gpx_path=gpx_path, output_path=DATA_BASE_PATH / "map_matched.json"
)

Map matching result saved to /home/jose_edsouza/Documentos/Faculdade/TCC/repo/dataset/newson-krumm/data/map_matched.json


In [14]:
grapphopper_result_points = polyline.decode(graphhopper_result["points"])

grapphopper_result_points[:5]

[(47.66753, -122.10708),
 (47.66753, -122.1054),
 (47.66756, -122.10458),
 (47.66754, -122.10237),
 (47.6676, -122.10215)]

In [15]:
measured_points_trajectory = list(gps_df.apply(lambda row: (row["lat"], row["lon"]), axis=1))
measured_points_trajectory[:5]

[(47.66748333, -122.1070833),
 (47.6675, -122.1070667),
 (47.6675, -122.1070667),
 (47.66751667, -122.1070333),
 (47.66753333, -122.1069833)]

In [16]:
grapphopper_trajectory = [res[-1] + 883000000000 for res in graphhopper_result["edge_ids"]]
grapphopper_trajectory[:5]

[884147800801, 884147800802, 884147800421, 884147800422, 884147800423]

In [17]:
ground_truth_trajectory = ground_truth_df["edge_id"].to_list()
ground_truth_trajectory[:5]

[884147800801, 884147800802, 884147800421, 884147800422, 884147800423]

In [18]:
len(grapphopper_trajectory), len(ground_truth_trajectory)

(584, 576)

In [19]:
from geopy.distance import great_circle
from geopy.point import Point
import pandas as pd

type Coordinate = tuple[float, float]  # (latitude, longitude)

def are_points_close(p1: Coordinate, p2: Coordinate, tolerance_meters: float = 0.1) -> bool:
    """Checks if two coordinates are within a specified distance (in meters)."""
    point1 = Point(p1[0], p1[1])
    point2 = Point(p2[0], p2[1])
    return great_circle(point1, point2).meters < tolerance_meters

def get_all_points_from_trajectory(
    trajectory: list[int], network_df: pd.DataFrame
) -> list[Coordinate]:
    """
    Given a list of edge_ids (trajectory) and a network DataFrame,
    returns the full sequence of connected coordinates, ensuring correct direction.
    """
    if not trajectory:
        return []

    all_points: list[Coordinate] = []

    for i, edge_id in enumerate(trajectory):
        segment_points: list[Coordinate] = network_df.loc[network_df["edge_id"] == edge_id, "track_segs"].values[0]

        if not segment_points:
            print(f"Warning: Edge {edge_id} has no track segments.")
            continue

        if i == 0:
            all_points.extend(segment_points)
        else:
            previous_last = all_points[-1]
            first_current = segment_points[0]
            last_current = segment_points[-1]

            # Check connection with tolerance
            if are_points_close(previous_last, first_current):
                # Correct direction, append from the second point
                all_points.extend(segment_points[1:])
            elif are_points_close(previous_last, last_current):
                # Reverse segment and append from the second point
                all_points.extend(segment_points[::-1][1:])
            else:
                print(f"Warning: Segment {edge_id} is not directly connected to previous segment {trajectory[i-1]}.")
                # Optional: Append all points of the current segment if not connected,
                # or handle as an error. For now, we'll append to see the structure.
                all_points.extend(segment_points) # Decide how to handle this disconnected part

    return all_points

In [20]:
gh_points_from_trajectory = get_all_points_from_trajectory(grapphopper_trajectory, route_network_df)
gt_points_from_trajectory = get_all_points_from_trajectory(ground_truth_trajectory, route_network_df)

In [21]:
def calculate_distance(segs_id: int, rn_df: pd.DataFrame = route_network_df) -> float:
    seg = rn_df[rn_df["edge_id"].isin([segs_id])]["track_segs"].values

    if len(seg) != 1:
        raise ValueError(f"Expected one segment for edge_id {segs_id}, got {len(seg)}")
    
    seg = seg[0]

    total_distance = 0.0
    for i in range(len(seg) - 1):
        total_distance += geodesic(seg[i], seg[i + 1]).meters
        
    return total_distance

In [22]:
added = [seg for seg in grapphopper_trajectory if seg not in ground_truth_trajectory]
removed = [seg for seg in ground_truth_trajectory if seg not in grapphopper_trajectory]

print(f"Added segments: {len(added)}")
print(f"Removed segments: {len(removed)}")

Added segments: 16
Removed segments: 8


In [23]:
ground_truth_distance = sum(
    calculate_distance(seg_id, route_network_df) for seg_id in ground_truth_trajectory
)

added_distance = sum(
    calculate_distance(seg_id, route_network_df) for seg_id in added
)
removed_distance = sum(
    calculate_distance(seg_id, route_network_df) for seg_id in removed
)

print(f"Ground truth distance: {ground_truth_distance:.2f} m")
print(f"Added distance: {added_distance:.2f} m")
print(f"Removed distance: {removed_distance:.2f} m")

Ground truth distance: 80241.64 m
Added distance: 882.14 m
Removed distance: 597.60 m


In [24]:
def calculate_accuracy(gt_dist: float, added_dist: float, removed_dist: float) -> float:
    if gt_dist == 0:
        return 0.0
    return 1 - ((added_dist + removed_dist) / gt_dist)

In [25]:
accuracy = calculate_accuracy(
    ground_truth_distance, added_distance, removed_distance
)

print(f"Accuracy: {accuracy:.2%}")

Accuracy: 98.16%


In [26]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


def plot_trajectories(
    original: list[Coordinate],
    calculated: list[Coordinate],
    title: str = "Tracks",
    original_label: str = "Original",
    calculated_label: str = "Calculated"
) -> None:
    # Criar DataFrame com ambas as trajetórias
    df = pd.DataFrame(
        [(lat, lon, original_label, i) for i, (lat, lon) in enumerate(original)] +
        [(lat, lon, calculated_label, i) for i, (lat, lon) in enumerate(calculated)],
        columns=['lat', 'lon', 'type', 'row_num']
    )

    # Inicializar figura vazia
    fig = go.Figure()

    # Adicionar pontos e linhas para cada tipo
    for type_ in [original_label, calculated_label]:
        subset = df[df['type'] == type_].sort_values('row_num')

        # Pontos
        fig.add_trace(go.Scattermap(
            lat=subset['lat'],
            lon=subset['lon'],
            mode='markers',
            marker=dict(size=10),
            name=f'{type_} - pontos',
            legendgroup=type_,
            visible=True
        ))

        # Linhas
        fig.add_trace(go.Scattermap(
            lat=subset['lat'],
            lon=subset['lon'],
            mode='lines',
            line=dict(width=2),
            name=f'{type_} - linha',
            legendgroup=type_,
            visible=True
        ))

    # Botões de controle de visibilidade
    fig.update_layout(
        updatemenus=[dict(
            type="buttons",
            direction="right",
            showactive=True,
            x=0.5,
            xanchor="center",
            y=1,  # move buttons above the title
            yanchor="top",
            buttons=[
                dict(label="Mostrar Nenhuma",
                     method="update",
                     args=[{"visible": [False, False, False, False]}]),

                dict(label="Mostrar Ambas",
                     method="update",
                     args=[{"visible": [True, True, True, True]}]),

                dict(label=f"Apenas {original_label}",
                     method="update",
                     args=[{"visible": [True, True, False, False]}]),

                dict(label=f"Apenas {calculated_label}",
                     method="update",
                     args=[{"visible": [False, False, True, True]}]),
            ]
        )]
    )

    # Estilo do mapa e título
    fig.update_layout(
        mapbox_style="open-street-map",
        mapbox=dict(
            center=dict(lat=df['lat'].mean(), lon=df['lon'].mean()),
            zoom=14
        ),
        margin=dict(l=0, r=0, t=80, b=0),  # increase top margin for buttons
        height=700,
        title=title
    )

    fig.show()


In [27]:
plot_trajectories(
    original=measured_points_trajectory,
    calculated=grapphopper_result_points,
    original_label="Mensurado",
    calculated_label="Grapphopper",
    title="Trajetórias GPS Mensuradas vs. Map Matching (Pontos)",
)

In [28]:
plot_trajectories(
    original=gt_points_from_trajectory,
    calculated=gh_points_from_trajectory,
    original_label="Ground Truth",
    calculated_label="Graphhopper",
    title="Trajetórias Ground Truth vs. Map Matching (Rede de Estradas)",
)

In [29]:
plot_trajectories(
    original=measured_points_trajectory,
    calculated=gh_points_from_trajectory,
    original_label="Mensurado",
    calculated_label="Grapphopper",
    title="Trajetórias GPS Mensuradas vs. Map Matching (Pontos x Rede de Estradas)",
)